# Unsupervised Ellipsoid Fitting Algorithm

This notebook extends the hypersphere-based algorithm by replacing each spherical region with an ellipsoid. The motivation is that normal embeddings in DINOv2 feature space are unlikely to form locally isotropic clusters. Instead, neighbourhoods may stretch more strongly along some directions than others. Ellipsoids are therefore able to model local variance more naturally than hyperspheres.

The ellipsoid formulation has several advantages:

- Aligns each region with the natural variance structure of the local KNN neighbourhood.
- Reduces unused empty space compared with hyperspheres, since the boundary can contract along low-variance directions.
- It can reduce unnecessary overlap between neighbouring regions by following the dominant principal axes of the local embedding distribution.
- It is better suited to high-variance categories, where the normal embedding space may contain elongated or anisotropic regions.
- Provides additional interpretability through eigenvalues, eigenvectors, axis ratios, and local region structure.

Several changes were introduced compared with the hypersphere version:

- Growth is variance-scaled rather than uniform. Expansion along each axis is controlled by the relative eigenvalue contribution, so high-variance directions can grow more than low-variance directions.
- Candidate cleaning is weight-based. Instead of immediately removing a point when a candidate ellipsoid overlaps a previous region, the algorithm first reduces that point’s contribution to the ellipsoid  fit. If its weight reaches zero and overlap remains, the point is removed.
- Sparse ellipsoids require additional support. Unlike hyperspheres, ellipsoids fitted from very few points can become geometrically unstable. To address this, the covariance of a small candidate region is blended with covariance information from a previous ellipsoid.
- The current support strategy borrows covariance from the nearest ellipsoid by centre distance. This is a limitation, since the nearest ellipsoid may not be the most geometrically similar. Future work should select support using both spatial proximity and shape similarity.

In [1]:
import os

import torch
import sqlite3
import pandas as pd

import json

from datetime import datetime
from dataclasses import asdict

from pathlib import Path
import sys 

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, EXPERIMENTS, RESULTS, DB_PATH

EMBED_PATH = EMBEDS_DIR / "base_embeds"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = EXPERIMENTS / EMBED_NAME / "ellipsoid"
RESULTS_DIR = RESULTS / EMBED_NAME

In [2]:
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

In [3]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)

In [4]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [5]:
%load_ext autoreload
%autoreload 2

from src.algorithims.ellipsoid import EllipsoidFitter, EllipsoidCover, CandidateCleaner, EllipsoidEvaluator
from src.types import ExperimentConfig, AlgorithmResults

metadata = ExperimentConfig(
    K_frac=0.05,
    start_growth=1.2,
    min_growth=1,
    reg=1e-4,
    growth_type="variance_scaled",
    cleaner="shared_axis"
)

fitter = EllipsoidFitter(support_points=5, reg=metadata.reg)
cleaner = CandidateCleaner(min_points=1, fitter=fitter)

cover = EllipsoidCover(fitter=fitter, cleaner=cleaner)

eval = EllipsoidEvaluator()

In [6]:
time = datetime.now().strftime("%y-%m-%d_%H-%M-%S")
aurocs = {}

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category / time
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[~train_mask]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    ellipsoids, ellipsoids_df = cover.run(
        embeds=cat_emb, output_dir=outputs_dir, 
        k_frac=metadata.K_frac, 
        start_growth=metadata.start_growth, min_growth=metadata.min_growth
        )
    
    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[~train_mask]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    overlaps_df, num_overlaps = eval.overlap(cat_emb, ellipsoids)
    overlaps_df.to_csv(outputs_dir / f"overlaps.csv", index=False)

    good_any, good_counts = eval.inside_any_count(good_test_emb, ellipsoids)
    defect_any, defect_counts = eval.inside_any_count(defect_test_emb, ellipsoids)

    results_df, metrics = eval.evaluate_detection(good_test_emb, defect_test_emb, ellipsoids)
    results_df.to_csv(outputs_dir / f"results.csv", index=False)

    diagnostics = eval.bucket_diagnostics(results_df)

    aurocs[category] = metrics["auroc"]

    results = AlgorithmResults(
        config=metadata,
        category=category,
        n_shapes=len(ellipsoids),
        auroc=metrics["auroc"],
        normal_inside=int(good_any.sum()),
        defect_inside=int(defect_any.sum())
    )

    with open(outputs_dir / f"metadata.json", "w") as f:
        json.dump(asdict(results), f)

aurocs_df = pd.DataFrame(aurocs.items(), columns=["Category", "AUROC"])

Running bottle
Running cable
Running capsule
Running carpet
Running grid
Running hazelnut
Running leather
Running metal_nut
Running pill
Running screw
Running tile
Running toothbrush
Running transistor
Running wood
Running zipper


In [7]:
df_roc_stats = pd.DataFrame({
    "mean": aurocs_df["AUROC"].mean(),
    "median": aurocs_df["AUROC"].median(),
    "std": aurocs_df["AUROC"].std(),
    "min_cat":  aurocs_df["Category"][aurocs_df["AUROC"].idxmin()],
    "min": aurocs_df["AUROC"].min(),
    "max_cat": aurocs_df["Category"][aurocs_df["AUROC"].idxmax()],
    "max": aurocs_df["AUROC"].max()
}, index=[0]).round(3)

df_roc_stats.to_csv(RESULTS_DIR / "ellipsoid_auroc_stats.csv", index=False)

aurocs_df = aurocs_df.round(3)
aurocs_df.to_csv(RESULTS_DIR /"ellipsoid_aurocs.csv", index=False)

df_roc_stats

,mean,median,std,min_cat,min,max_cat,max
0,0.901,0.934,0.099,transistor,0.688,bottle,1.0


In [ ]:
display(diagnostics["n_points"])
display(diagnostics["eig_ratio"])
display(diagnostics["n_points_by_class"])
display(diagnostics["eig_ratio_aurocs"])